# H8 HazardTwin · Galați (Dunăre) · CP3: învățare, rezultate v1

Detecția inundației pe imagine din **Sentinel-1 (VV/VH)**, pe SEN12FLOOD (Rambour et al., 2020).
Comparăm metoda propusă (**M2: gradient boosting pe trăsături de imagine + anomalie față de imaginile anterioare din serie**) cu baseline-uri: majoritar, prag Otsu global (o imagine), prag pe schimbare (serie) și GBM fără timp.

Evaluare: split **spațial** pe regiuni de 2° × 2°, repetat pe 5 seed-uri, cu IC 95 % (t) peste seed-uri și diferențe pereche. Split-ul oficial (cu scurgere spațială) e raportat o singură dată, pentru comparație.

Tot codul proiectului e scris mai jos cu `%%writefile` (identic cu repo-ul [IchimCosmin/h8-hazardtwin-galati](https://github.com/IchimCosmin/h8-hazardtwin-galati)). Nu are nevoie de internet; datele vin din dataset-ul Kaggle *SEN12FLOOD: Flood Detection Dataset*.

In [ ]:
import glob, os, sys, time
SRC = '/kaggle/working/src'
OUT = '/kaggle/working/out'
os.makedirs(SRC + '/h8twin', exist_ok=True)
os.makedirs(OUT, exist_ok=True)
sys.path.insert(0, SRC)
cands = sorted(glob.glob('/kaggle/input/**/S1list.json', recursive=True), key=len)
ROOT = os.path.dirname(cands[0])
print('SEN12FLOOD root:', ROOT, '| folders:', len(os.listdir(ROOT)), '| CPUs:', os.cpu_count())

In [ ]:
%%writefile {SRC}/h8twin/__init__.py
"""H8 HazardTwin — Galați Danube flood twin: data ingestion utilities."""
__version__ = "0.1.0"


In [ ]:
%%writefile {SRC}/h8twin/sen12flood.py
"""SEN12FLOOD ingestion: metadata index, image loading, sequence tensors, splits.

The dataset (Rambour et al., 2020; doi:10.21227/w6xz-s898) ships as
    <root>/S1list.json, <root>/S2list.json, <root>/0001 ... <root>/0336
Each JSON maps a sequence id to a dict holding sequence-level fields
(image count, folder, geography) and one dict per image with the acquisition
date, FLOODING, FULL-DATA-COVERAGE, a file prefix and, for S1, the orbit.

The parser is deliberately tolerant about key spelling so that small
differences between dataset mirrors do not break ingestion.
"""
from __future__ import annotations

import json
import re
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable

import numpy as np
import pandas as pd

S2_BANDS = ["B01", "B02", "B03", "B04", "B05", "B06", "B07", "B08", "B8A", "B09", "B11", "B12"]
S1_POLS = ["VV", "VH"]


# --------------------------------------------------------------------------- helpers
def _norm(key: str) -> str:
    return re.sub(r"[^a-z0-9]", "", str(key).lower())


def _get(d: dict, *names, default=None):
    """Case/punctuation-insensitive dict lookup."""
    table = {_norm(k): v for k, v in d.items()}
    for n in names:
        v = table.get(_norm(n))
        if v is not None:
            return v
    return default


def _to_bool(v) -> bool | None:
    if v is None:
        return None
    if isinstance(v, bool):
        return v
    s = str(v).strip().lower()
    if s in {"true", "1", "yes", "t"}:
        return True
    if s in {"false", "0", "no", "f"}:
        return False
    return None


def _is_image_entry(v) -> bool:
    return isinstance(v, dict) and any("flood" in _norm(k) for k in v)


# --------------------------------------------------------------------------- index
def parse_list(json_path: Path, sensor: str) -> pd.DataFrame:
    """Flatten S1list.json / S2list.json into one row per image."""
    with open(json_path, encoding="utf-8") as f:
        raw = json.load(f)
    seqs = raw.items() if isinstance(raw, dict) else enumerate(raw)
    rows = []
    for seq_key, seq in seqs:
        if not isinstance(seq, dict):
            continue
        folder = str(_get(seq, "folder", "foldername", "id", default=seq_key))
        geography = _get(seq, "geography", "geo", "geometry", "location")
        n_declared = _get(seq, "count", "nimages", "number", "total", "length")
        for img_key, img in seq.items():
            if not _is_image_entry(img):
                continue
            rows.append({
                "sensor": sensor,
                "seq": folder,
                "img_key": str(img_key),
                "date": pd.to_datetime(_get(img, "date", "acquisitiondate", "datetime"), errors="coerce"),
                "flooding": _to_bool(_get(img, "FLOODING", "flood", "flooded")),
                "full_coverage": _to_bool(_get(img, "FULL-DATA-COVERAGE", "fulldatacoverage", "coverage")),
                "prefix": _get(img, "filename", "prefix", "file", "name"),
                "orbit": _get(img, "orbit", "orbitdirection") if sensor == "S1" else None,
                "geography": json.dumps(geography) if isinstance(geography, (dict, list)) else geography,
                "n_declared": int(n_declared) if str(n_declared).isdigit() else None,
            })
    df = pd.DataFrame(rows)
    if not df.empty:
        df = df.sort_values(["seq", "date"]).reset_index(drop=True)
        # the IEEE DataPort release keeps the original split in the folder names:
        # 4-digit folders (0001..0336) = train, unpadded folders (0..67) = test
        df["official_split"] = np.where(df["seq"].str.fullmatch(r"\d{4}"), "train", "test")
    return df


def find_files(root: Path, row: pd.Series) -> dict[str, Path]:
    """Map band/polarisation -> GeoTIFF path for one image row."""
    folder = root / row["seq"]
    prefix = str(row["prefix"] or "")
    cands = sorted(folder.glob(f"{prefix}*.tif")) if prefix else []
    keys = S1_POLS if row["sensor"] == "S1" else S2_BANDS
    out = {}
    for k in keys:
        hit = [p for p in cands if re.search(rf"(^|[_\-.]){k}([_\-.]|$)", p.stem, re.I)]
        if hit:
            out[k] = hit[0]
    return out


def build_index(root: str | Path, sensors: Iterable[str] = ("S1", "S2")) -> pd.DataFrame:
    """One row per image, with file paths and an `available` flag."""
    root = Path(root)
    frames = []
    for s in sensors:
        jp = root / f"{s}list.json"
        if not jp.exists():
            raise FileNotFoundError(f"missing {jp}")
        df = parse_list(jp, s)
        files = df.apply(lambda r: find_files(root, r), axis=1)
        keys = S1_POLS if s == "S1" else S2_BANDS
        df["n_files"] = files.map(len)
        df["available"] = df["n_files"] == len(keys)
        df["files"] = files.map(lambda d: json.dumps({k: str(v) for k, v in d.items()}))
        frames.append(df)
    return pd.concat(frames, ignore_index=True)


# --------------------------------------------------------------------------- pixels
def read_s1(paths: dict[str, Path | str], to_db: bool = True) -> np.ndarray:
    """(2, H, W) float32 VV/VH. Linear sigma0 is converted to dB; nodata -> NaN."""
    import rasterio

    arrs = []
    for pol in S1_POLS:
        with rasterio.open(paths[pol]) as src:
            a = src.read(1).astype("float32")
            if src.nodata is not None:
                a[a == src.nodata] = np.nan
        arrs.append(a)
    h = min(a.shape[0] for a in arrs)
    w = min(a.shape[1] for a in arrs)
    x = np.stack([a[:h, :w] for a in arrs])
    x[~np.isfinite(x)] = np.nan
    if to_db and is_linear(x):
        x[x <= 0] = np.nan
        x = 10.0 * np.log10(x)
    return x


def is_linear(x: np.ndarray) -> bool:
    """Heuristic: sigma0 in linear power is mostly in (0, 1]; in dB it is mostly negative."""
    v = x[np.isfinite(x)]
    if v.size == 0:
        return False
    return float(np.nanmedian(v)) > 0 and float(np.nanpercentile(v, 95)) < 5


@dataclass
class Sequence:
    seq: str
    dates: list
    x: np.ndarray        # (T, 2, H, W) dB
    y: np.ndarray        # (T,) 0/1
    orbit: list


def load_sequence(index: pd.DataFrame, seq: str, size: int | None = 256,
                  only_full: bool = False) -> Sequence:
    """Stack the S1 time series of one sequence into a tensor (centre-cropped to `size`)."""
    rows = index[(index.sensor == "S1") & (index.seq == seq) & index.available]
    if only_full:
        rows = rows[rows.full_coverage.fillna(False)]
    rows = rows.sort_values("date")
    xs, ys, ds, ob = [], [], [], []
    for _, r in rows.iterrows():
        x = read_s1(json.loads(r["files"]))
        xs.append(x)
        ys.append(int(bool(r["flooding"])))
        ds.append(r["date"])
        ob.append(r["orbit"])
    if not xs:
        raise ValueError(f"no S1 images available for sequence {seq}")
    h = min(a.shape[1] for a in xs)
    w = min(a.shape[2] for a in xs)
    if size:
        h, w = min(h, size), min(w, size)
    def crop(a):
        t, l = (a.shape[1] - h) // 2, (a.shape[2] - w) // 2
        return a[:, t:t + h, l:l + w]
    return Sequence(seq, ds, np.stack([crop(a) for a in xs]).astype("float32"),
                    np.array(ys, dtype="int8"), ob)


# --------------------------------------------------------------------------- splits
def split_sequences(index: pd.DataFrame, test_frac: float = 0.2, seed: int = 42,
                    group_by: str = "region") -> pd.DataFrame:
    """Sequence-level split; images of one sequence never cross train/test.

    group_by="region" keeps whole geographic clusters (rounded centroid, ~2 deg)
    on one side, the spatial split the course asks for; "seq" splits by sequence only.
    """
    seqs = index.groupby("seq").agg(geography=("geography", "first")).reset_index()
    if group_by == "region":
        seqs["group"] = seqs["geography"].map(_region_key)
        seqs.loc[seqs.group.isna(), "group"] = seqs["seq"]
    else:
        seqs["group"] = seqs["seq"]
    groups = seqs["group"].drop_duplicates().sample(frac=1.0, random_state=seed).tolist()
    n_test_target = int(round(test_frac * len(seqs)))
    test, n = set(), 0
    sizes = seqs.group.value_counts()
    for g in groups:  # greedy fill, skipping groups that would overshoot the target by >15 %
        if n >= n_test_target:
            break
        if n + sizes[g] > 1.15 * n_test_target and n > 0:
            continue
        test.add(g)
        n += int(sizes[g])
    seqs["split"] = np.where(seqs.group.isin(test), "test", "train")
    return seqs[["seq", "group", "split"]]


def centroid(geo) -> tuple[float, float] | None:
    """(lon, lat) mean of all coordinates of a GeoJSON-like geometry."""
    try:
        g = json.loads(geo) if isinstance(geo, str) else geo
    except (TypeError, ValueError):
        return None
    coords = []

    def walk(o):
        if isinstance(o, (list, tuple)) and len(o) == 2 and all(isinstance(c, (int, float)) for c in o):
            coords.append(o)
        elif isinstance(o, (list, tuple)):
            for c in o:
                walk(c)
        elif isinstance(o, dict):
            for c in o.values():
                walk(c)
    walk(g)
    if not coords:
        return None
    lon, lat = np.mean(np.array(coords, dtype=float), axis=0)
    return float(lon), float(lat)


def official_split_leakage(index: pd.DataFrame) -> dict:
    """How close the official test tiles are to train tiles, and how many S1 scenes they share."""
    s1 = index[index.sensor == "S1"]
    seqs = s1.groupby("seq").agg(geo=("geography", "first"), split=("official_split", "first"))
    cen = seqs.geo.map(centroid)
    seqs = seqs[cen.notna()].assign(lon=cen.dropna().map(lambda c: c[0]), lat=cen.dropna().map(lambda c: c[1]))
    tr, te = seqs[seqs.split == "train"], seqs[seqs.split == "test"]
    if tr.empty or te.empty:
        return {}
    d = []
    for _, r in te.iterrows():
        dx = (tr.lon - r.lon) * 111.32 * np.cos(np.radians(r.lat))
        dy = (tr.lat - r.lat) * 110.57
        d.append(float(np.sqrt(dx ** 2 + dy ** 2).min()))
    d = np.array(d)
    ptr = set(s1[s1.official_split == "train"].prefix)
    pte = set(s1[s1.official_split == "test"].prefix)
    return {"official_train_seqs": int(len(tr)), "official_test_seqs": int(len(te)),
            "test_to_nearest_train_km_median": round(float(np.median(d)), 1),
            "test_within_10km_of_train": int((d < 10).sum()),
            "test_s1_scenes_also_in_train": int(len(pte & ptr)), "test_s1_scenes": int(len(pte))}


def _region_key(geo, deg: float = 2.0):
    """Round the centroid of a GeoJSON-like geometry (or lon/lat pair) to a coarse grid cell."""
    if geo is None or (isinstance(geo, float) and np.isnan(geo)):
        return None
    try:
        g = json.loads(geo) if isinstance(geo, str) else geo
    except (TypeError, ValueError):
        return None
    coords = []

    def walk(o):
        if isinstance(o, (list, tuple)) and len(o) == 2 and all(isinstance(c, (int, float)) for c in o):
            coords.append(o)
        elif isinstance(o, (list, tuple)):
            for c in o:
                walk(c)
        elif isinstance(o, dict):
            for c in o.values():
                walk(c)
    walk(g)
    if not coords:
        return None
    lon, lat = np.mean(np.array(coords, dtype=float), axis=0)
    return f"{np.floor(lat / deg) * deg:+.0f}_{np.floor(lon / deg) * deg:+.0f}"


In [ ]:
%%writefile {SRC}/h8twin/stats.py
"""Numbers for the data card, computed from the index (and a pixel sample)."""
from __future__ import annotations

import json

import numpy as np
import pandas as pd

from .sen12flood import read_s1


def _gaps_days(dates: pd.Series) -> np.ndarray:
    d = pd.to_datetime(dates.dropna()).sort_values().drop_duplicates()
    return d.diff().dt.days.dropna().to_numpy()


def index_stats(index: pd.DataFrame) -> dict:
    out = {}
    for s, df in index.groupby("sensor"):
        per_seq = df.groupby("seq").size()
        gaps = np.concatenate([_gaps_days(g.date) for _, g in df.groupby("seq")] or [np.array([])])
        flood_seq = df.groupby("seq").flooding.apply(lambda v: bool(v.fillna(False).any()))
        out[s] = {
            "sequences": int(df.seq.nunique()),
            "images_listed": int(len(df)),
            "images_with_all_files": int(df.available.sum()),
            "images_missing_files": int((~df.available).sum()),
            "images_per_seq_min_median_max": [int(per_seq.min()), float(per_seq.median()), int(per_seq.max())],
            "date_min": str(df.date.min().date()) if df.date.notna().any() else None,
            "date_max": str(df.date.max().date()) if df.date.notna().any() else None,
            "pct_images_flooded": round(100 * df.flooding.fillna(False).mean(), 1),
            "pct_sequences_with_flood": round(100 * flood_seq.mean(), 1),
            "pct_full_coverage": (None if df.full_coverage.isna().all()
                                  else round(100 * df.full_coverage.fillna(False).mean(), 1)),
            "revisit_days_median": float(np.median(gaps)) if gaps.size else None,
            "revisit_days_p90": float(np.percentile(gaps, 90)) if gaps.size else None,
            "revisit_days_max": float(gaps.max()) if gaps.size else None,
            "missing_dates": int(df.date.isna().sum()),
            "missing_labels": int(df.flooding.isna().sum()),
        }
        if s == "S1":
            out[s]["orbit_counts"] = {str(k): int(v) for k, v in df.orbit.value_counts(dropna=False).items()}
    return out


def raster_profile(path: str) -> dict:
    import rasterio

    with rasterio.open(path) as src:
        res = src.res
        return {"crs": str(src.crs), "width": src.width, "height": src.height,
                "res": [float(res[0]), float(res[1])], "dtype": src.dtypes[0],
                "count": src.count, "nodata": src.nodata}


def pixel_stats(index: pd.DataFrame, n: int = 60, seed: int = 0) -> dict:
    """Backscatter (dB) statistics on a random sample of available S1 images, split by label."""
    s1 = index[(index.sensor == "S1") & index.available]
    if s1.empty:
        return {}
    sample = s1.sample(min(n, len(s1)), random_state=seed)
    vals = {True: {"VV": [], "VH": []}, False: {"VV": [], "VH": []}}
    nan_frac = []
    for _, r in sample.iterrows():
        x = read_s1(json.loads(r["files"]))
        nan_frac.append(float(np.isnan(x).mean()))
        lab = bool(r["flooding"])
        for i, pol in enumerate(["VV", "VH"]):
            v = x[i][np.isfinite(x[i])]
            if v.size:
                vals[lab][pol].append(np.median(v))
    res = {"images_sampled": int(len(sample)), "nodata_pct_mean": round(100 * float(np.mean(nan_frac)), 2)}
    for lab, name in [(True, "flooded"), (False, "not_flooded")]:
        for pol in ["VV", "VH"]:
            a = np.array(vals[lab][pol])
            res[f"median_{pol}_dB_{name}"] = round(float(np.median(a)), 2) if a.size else None
    first = json.loads(sample.iloc[0]["files"])
    res["profile_VV"] = raster_profile(first["VV"])
    return res


In [ ]:
%%writefile {SRC}/h8twin/features.py
"""Per-image features for SEN12FLOOD S1 images (CP3).

Two families, both computed from VV/VH backscatter in dB:

* single-image ("instant"): distribution of VV, VH and VV-VH, dark-pixel
  fractions, plus a VV histogram used by the global-Otsu baseline;
* temporal ("anomaly vs. normal"): each image is compared with the per-pixel
  median of the *previous* images of the same sequence and orbit direction.
  Only the past is used, so the same features can run in real time on Galați.
"""
from __future__ import annotations

import json

import numpy as np
import pandas as pd

from .sen12flood import read_s1

HIST_EDGES = np.arange(-35.0, 10.0 + 0.25, 0.25)       # 180 bins for VV histograms
DARK_VV = (-15, -18, -20, -22)
DARK_VH = (-22, -25, -28)
Q = (5, 25, 50, 75, 95)


def _q(v, prefix):
    if v.size == 0:
        return {f"{prefix}_q{q}": np.nan for q in Q} | {f"{prefix}_mean": np.nan, f"{prefix}_std": np.nan}
    qs = np.percentile(v, Q)
    out = {f"{prefix}_q{q}": float(x) for q, x in zip(Q, qs)}
    out[f"{prefix}_mean"] = float(v.mean())
    out[f"{prefix}_std"] = float(v.std())
    return out


def instant_features(x: np.ndarray) -> tuple[dict, np.ndarray]:
    """x: (2, H, W) dB. Returns (features, VV histogram counts)."""
    vv, vh = x[0], x[1]
    ok = np.isfinite(vv) & np.isfinite(vh)
    f = {"valid_frac": float(ok.mean())}
    v, h = vv[ok], vh[ok]
    f |= _q(v, "vv") | _q(h, "vh") | _q(v - h, "ratio")
    for t in DARK_VV:
        f[f"vv_lt{-t}"] = float((v < t).mean()) if v.size else np.nan
    for t in DARK_VH:
        f[f"vh_lt{-t}"] = float((h < t).mean()) if h.size else np.nan
    hist, _ = np.histogram(np.clip(v, HIST_EDGES[0], HIST_EDGES[-1] - 1e-6), bins=HIST_EDGES)
    return f, hist.astype("int64")


def temporal_features(x: np.ndarray, ref: np.ndarray | None, n_prev: int) -> dict:
    """Compare x with the per-pixel median `ref` of earlier images (same orbit)."""
    f = {"n_prev": n_prev}
    keys = ["d_vv_mean", "d_vv_q5", "d_vv_q50", "d_vh_mean", "d_vh_q5", "d_vh_q50",
            "d_vv_lt3", "d_vv_lt6", "d_vh_lt3", "d_vh_lt6", "new_dark_vv18", "new_dark_vh25"]
    if ref is None:
        return f | {k: np.nan for k in keys}
    d = x - ref
    ok = np.isfinite(d[0]) & np.isfinite(d[1])
    dv, dh = d[0][ok], d[1][ok]
    if dv.size == 0:
        return f | {k: np.nan for k in keys}
    f |= {"d_vv_mean": float(dv.mean()), "d_vv_q5": float(np.percentile(dv, 5)),
          "d_vv_q50": float(np.median(dv)), "d_vh_mean": float(dh.mean()),
          "d_vh_q5": float(np.percentile(dh, 5)), "d_vh_q50": float(np.median(dh)),
          "d_vv_lt3": float((dv < -3).mean()), "d_vv_lt6": float((dv < -6).mean()),
          "d_vh_lt3": float((dh < -3).mean()), "d_vh_lt6": float((dh < -6).mean())}
    vv, rvv, vh, rvh = x[0][ok], ref[0][ok], x[1][ok], ref[1][ok]
    f["new_dark_vv18"] = float(((vv < -18) & (rvv >= -18)).mean())
    f["new_dark_vh25"] = float(((vh < -25) & (rvh >= -25)).mean())
    return f


def sequence_features(rows: pd.DataFrame, max_ref: int = 6) -> tuple[list[dict], list[np.ndarray]]:
    """Features for all available S1 images of one sequence, in date order."""
    rows = rows.sort_values("date")
    imgs = [read_s1(json.loads(r)) for r in rows["files"]]
    h = min(a.shape[1] for a in imgs)
    w = min(a.shape[2] for a in imgs)
    imgs = [a[:, :h, :w] for a in imgs]
    feats, hists = [], []
    for i, (_, r) in enumerate(rows.iterrows()):
        f, hist = instant_features(imgs[i])
        prev = [j for j in range(i) if rows.iloc[j]["orbit"] == r["orbit"]][-max_ref:]
        ref = np.nanmedian(np.stack([imgs[j] for j in prev]), axis=0) if prev else None
        f |= temporal_features(imgs[i], ref, len(prev))
        f |= {"seq": r["seq"], "img_key": r["img_key"], "date": r["date"], "orbit": r["orbit"],
              "flooding": int(bool(r["flooding"])), "official_split": r["official_split"],
              "geography": r["geography"], "t_index": i}
        feats.append(f)
        hists.append(hist)
    return feats, hists


def extract_features(index: pd.DataFrame, n_jobs: int = -1, verbose: int = 0):
    """Run `sequence_features` over all sequences. Returns (features DataFrame, VV histograms)."""
    from joblib import Parallel, delayed

    s1 = index[(index.sensor == "S1") & index.available]
    groups = [g for _, g in s1.groupby("seq")]
    out = Parallel(n_jobs=n_jobs, verbose=verbose)(delayed(sequence_features)(g) for g in groups)
    feats = [f for fs, _ in out for f in fs]
    hists = [h for _, hs in out for h in hs]
    df = pd.DataFrame(feats)
    return df.reset_index(drop=True), np.stack(hists)


In [ ]:
%%writefile {SRC}/h8twin/learn.py
"""CP3 experiment: method vs. baselines on SEN12FLOOD S1, spatial splits over seeds.

Task: per-image flood label (FLOODING) from Sentinel-1 VV/VH.

Methods
  B0  majority class                                   (no learning)
  B1  global Otsu threshold on VV + tuned area cut-off (single image, SOA baseline)
  B2  change threshold: share of pixels with VV drop > 3 dB vs. earlier images
  M1  gradient boosting on single-image features       (learning, no time)
  M2  gradient boosting on single-image + anomaly-vs-past features (proposed)

Evaluation: region-grouped spatial split (2 deg cells), repeated over seeds;
95 % CI across seeds (t distribution) and paired differences vs. the baselines.
The leaky official split is run once, for comparison with Rambour et al. (2020).
"""
from __future__ import annotations

import numpy as np
import pandas as pd
from scipy import stats
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
                             precision_score, recall_score, roc_auc_score)
from sklearn.utils.class_weight import compute_sample_weight

from .features import HIST_EDGES
from .sen12flood import split_sequences

META = {"seq", "img_key", "date", "orbit", "flooding", "official_split", "geography", "t_index"}
TEMPORAL = {"n_prev", "d_vv_mean", "d_vv_q5", "d_vv_q50", "d_vh_mean", "d_vh_q5", "d_vh_q50",
            "d_vv_lt3", "d_vv_lt6", "d_vh_lt3", "d_vh_lt6", "new_dark_vv18", "new_dark_vh25"}
METHODS = ["B0 majoritar", "B1 prag Otsu (o imagine)", "B2 prag pe schimbare (serie)",
           "M1 GBM o imagine", "M2 GBM + anomalie temporală"]


def otsu_from_hist(hist: np.ndarray, edges: np.ndarray = HIST_EDGES) -> float:
    """Otsu threshold from a histogram (counts per bin)."""
    p = hist.astype(float) / max(hist.sum(), 1)
    centers = (edges[:-1] + edges[1:]) / 2
    w0 = np.cumsum(p)
    mu = np.cumsum(p * centers)
    mu_t = mu[-1]
    w1 = 1 - w0
    with np.errstate(divide="ignore", invalid="ignore"):
        between = (mu_t * w0 - mu) ** 2 / (w0 * w1)
    between[~np.isfinite(between)] = -1
    return float(edges[1:][np.argmax(between)])


def frac_below(hists: np.ndarray, thr: float, edges: np.ndarray = HIST_EDGES) -> np.ndarray:
    k = np.searchsorted(edges[1:], thr, side="right")
    tot = hists.sum(1)
    return hists[:, :k].sum(1) / np.maximum(tot, 1)


def tune_cutoff(score: np.ndarray, y: np.ndarray) -> float:
    """Cut-off on a 1-D score that maximises balanced accuracy on training data."""
    s = np.nan_to_num(score, nan=0.0)
    cands = np.unique(np.quantile(s, np.linspace(0, 1, 201)))
    best, best_t = -1, cands[0]
    for t in cands:
        b = balanced_accuracy_score(y, (s > t).astype(int))
        if b > best:
            best, best_t = b, t
    return float(best_t)


def metrics(y, pred, score) -> dict:
    out = {"bal_acc": balanced_accuracy_score(y, pred), "acc": accuracy_score(y, pred),
           "f1": f1_score(y, pred, zero_division=0), "precision": precision_score(y, pred, zero_division=0),
           "recall": recall_score(y, pred, zero_division=0)}
    try:
        out["auc"] = roc_auc_score(y, np.nan_to_num(score, nan=0.0))
    except ValueError:
        out["auc"] = np.nan
    return out


def feature_sets(df: pd.DataFrame) -> tuple[list[str], list[str]]:
    feats = [c for c in df.columns if c not in META]
    inst = [c for c in feats if c not in TEMPORAL]
    return inst, feats


def run_split(df: pd.DataFrame, hists: np.ndarray, is_test: np.ndarray, seed: int) -> list[dict]:
    tr, te = ~is_test, is_test
    y_tr, y_te = df.flooding.values[tr], df.flooding.values[te]
    rows = []

    # B0 majority
    maj = int(y_tr.mean() > 0.5)
    rows.append((METHODS[0], np.full(te.sum(), maj), np.full(te.sum(), 0.5)))

    # B1 global Otsu on pooled training histograms + tuned area cut-off
    thr = otsu_from_hist(hists[tr].sum(0))
    s_tr, s_te = frac_below(hists[tr], thr), frac_below(hists[te], thr)
    c = tune_cutoff(s_tr, y_tr)
    rows.append((METHODS[1], (s_te > c).astype(int), s_te))

    # B2 change threshold (DeVries-style), no learning beyond the cut-off
    s_tr, s_te = df.d_vv_lt3.values[tr], df.d_vv_lt3.values[te]
    c = tune_cutoff(s_tr, y_tr)
    rows.append((METHODS[2], (np.nan_to_num(s_te) > c).astype(int), s_te))

    # M1 / M2 gradient boosting, class-balanced
    inst, full = feature_sets(df)
    for name, cols in [(METHODS[3], inst), (METHODS[4], full)]:
        m = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, max_leaf_nodes=31,
                                           l2_regularization=1.0, random_state=seed)
        m.fit(df.loc[tr, cols], y_tr, sample_weight=compute_sample_weight("balanced", y_tr))
        p = m.predict_proba(df.loc[te, cols])[:, 1]
        rows.append((name, (p > 0.5).astype(int), p))

    out = []
    for name, pred, score in rows:
        r = {"method": name, "seed": seed, "n_test_img": int(te.sum()),
             "n_test_seq": int(df.seq[te].nunique()), "test_pos_rate": float(y_te.mean()),
             "otsu_thr_db": thr if name == METHODS[1] else np.nan}
        r |= metrics(y_te, pred, score)
        out.append(r)
    return out


def ci95(v: np.ndarray) -> tuple[float, float, float]:
    v = np.asarray(v, float)
    m = v.mean()
    if len(v) < 2:
        return m, np.nan, np.nan
    h = stats.t.ppf(0.975, len(v) - 1) * v.std(ddof=1) / np.sqrt(len(v))
    return m, m - h, m + h


def run_experiment(df: pd.DataFrame, hists: np.ndarray, seeds=(0, 1, 2, 3, 4),
                   test_frac: float = 0.2) -> pd.DataFrame:
    df = df.reset_index(drop=True)
    res = []
    for s in seeds:
        sp = split_sequences(df, test_frac=test_frac, seed=s).set_index("seq").split
        is_test = df.seq.map(sp).eq("test").values
        for r in run_split(df, hists, is_test, s):
            res.append({"row_type": "run", "split": "spatial_2deg"} | r)
    # leaky official split, once, for comparison with the literature
    is_test = df.official_split.eq("test").values
    if 0 < is_test.sum() < len(df):
        for r in run_split(df, hists, is_test, 0):
            res.append({"row_type": "run", "split": "official_leaky"} | r)
    runs = pd.DataFrame(res)

    summ = []
    sp = runs[runs.split == "spatial_2deg"]
    for name, g in sp.groupby("method", sort=False):
        for met in ["bal_acc", "f1", "auc", "acc"]:
            m, lo, hi = ci95(g[met].values)
            summ.append({"row_type": "summary", "split": "spatial_2deg", "method": name, "metric": met,
                         "mean": m, "ci95_lo": lo, "ci95_hi": hi, "n_seeds": len(g)})
    piv = sp.pivot(index="seed", columns="method", values="bal_acc")
    for a, b in [(METHODS[4], METHODS[1]), (METHODS[4], METHODS[3]), (METHODS[3], METHODS[1]),
                 (METHODS[4], METHODS[2])]:
        m, lo, hi = ci95((piv[a] - piv[b]).values)
        summ.append({"row_type": "paired_diff", "split": "spatial_2deg", "method": f"{a} − {b}",
                     "metric": "bal_acc", "mean": m, "ci95_lo": lo, "ci95_hi": hi, "n_seeds": len(piv)})
    return pd.concat([runs, pd.DataFrame(summ)], ignore_index=True)


In [ ]:
%%writefile {SRC}/h8twin/figures.py
"""CP1 figure: does the per-image FLOODING label show up in the S1 pixels?"""
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd

from .sen12flood import load_sequence

BLUE, ORANGE, INK, MUTED, GRID = "#3F6EA8", "#C8702A", "#1B2A41", "#6A7179", "#E3E0D8"


def dark_fraction(x: np.ndarray, thr_db: float) -> np.ndarray:
    """% of valid VV pixels below `thr_db`, per time step. x: (T, 2, H, W) in dB."""
    vv = x[:, 0].reshape(len(x), -1)
    valid = np.isfinite(vv)
    return 100 * np.where(valid, vv < thr_db, False).sum(1) / np.maximum(valid.sum(1), 1)


def cp1_figure(index: pd.DataFrame, out: Path, seq_main: str | None = None,
               thr_db: float = -18.0, lang: str = "ro") -> Path:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    from matplotlib import font_manager

    names = {f.name for f in font_manager.fontManager.ttflist}
    plt.rcParams.update({"font.family": "IBM Plex Sans" if "IBM Plex Sans" in names else "DejaVu Sans",
                         "font.size": 9, "axes.edgecolor": MUTED, "axes.labelcolor": INK,
                         "xtick.color": MUTED, "ytick.color": MUTED, "axes.titlecolor": INK})
    T = {"ro": dict(flood="etichetat inundat", dry="etichetat neinundat",
                    y=f"% pixeli VV < {thr_db:g} dB", a="a) Secvența {s}: apa „întunecată” crește după etichetă",
                    b="b) Mediana pe secvență: {k} din {n} cresc clar", seq="secvența"),
         "en": dict(flood="labelled flooded", dry="labelled not flooded",
                    y=f"% pixels VV < {thr_db:g} dB", a="a) Sequence {s}: dark water rises after the label",
                    b="b) Per-sequence median: {k} of {n} rise clearly", seq="sequence")}[lang]

    s1 = index[(index.sensor == "S1") & index.available]
    mixed = s1.groupby("seq").flooding.agg(lambda v: 0 < v.mean() < 1)
    seqs = [s for s in mixed[mixed].index]
    if not seqs:
        raise ValueError("no available sequence with both labels")
    seq_main = seq_main or max(seqs, key=lambda s: (s1.seq == s).sum())

    per_seq = {}
    for s in seqs:
        sq = load_sequence(index, s, size=None)
        per_seq[s] = (sq, dark_fraction(sq.x, thr_db))

    fig, (ax, bx) = plt.subplots(1, 2, figsize=(9.2, 3.3), dpi=220,
                                 gridspec_kw={"width_ratios": [1.75, 1]})
    sq, dk = per_seq[seq_main]
    pos = np.arange(len(dk))
    cols = [ORANGE if y else BLUE for y in sq.y]
    ax.bar(pos, dk, width=0.62, color=cols, edgecolor="white", linewidth=1.5)
    ax.set_xticks(pos, [pd.Timestamp(d).strftime("%d.%m") for d in sq.dates], rotation=0, fontsize=7.5)
    ax.set_ylabel(T["y"])
    ax.set_title(T["a"].format(s=seq_main), loc="left", fontsize=9.5, fontweight="bold")
    ax.bar([0], [0], color=BLUE, label=T["dry"])
    ax.bar([0], [0], color=ORANGE, label=T["flood"])
    ax.legend(frameon=False, fontsize=8, loc="upper left")
    ax.yaxis.grid(True, color=GRID, lw=0.8)
    ax.set_axisbelow(True)

    rows = sorted(per_seq, key=lambda s: str(s))
    k = 0
    for i, s in enumerate(rows):
        q, d = per_seq[s]
        pre, post = np.median(d[q.y == 0]), np.median(d[q.y == 1])
        k += int(post > 1.5 * pre + 0.5)
        bx.plot([pre, post], [i, i], color=GRID, lw=2.5, zorder=1)
        bx.scatter([post], [i], s=60, color=ORANGE, zorder=2, edgecolor="white", linewidth=1)
        bx.scatter([pre], [i], s=30, color=BLUE, zorder=3, edgecolor="white", linewidth=1)
    bx.set_yticks(range(len(rows)), [f"{T['seq']} {s}" for s in rows])
    bx.set_xlabel(T["y"])
    bx.set_title(T["b"].format(k=k, n=len(rows)), loc="left", fontsize=9.5, fontweight="bold")
    bx.xaxis.grid(True, color=GRID, lw=0.8)
    bx.set_axisbelow(True)
    bx.set_ylim(-0.6, len(rows) - 0.4)
    for a in (ax, bx):
        a.spines[["top", "right"]].set_visible(False)
    fig.tight_layout(w_pad=2.5)
    path = Path(out) / ("fig_cp1_sequence.png" if lang == "ro" else f"fig_cp1_sequence_{lang}.png")
    fig.savefig(path, facecolor="white")
    plt.close(fig)
    return path


In [ ]:
%%writefile {SRC}/h8twin/cli.py
"""Command-line entry point.

    python -m h8twin.cli ingest  --root D:/data/SEN12FLOOD --out outputs
    python -m h8twin.cli figure  --out outputs [--seq 1]

`ingest` writes outputs/index.csv, outputs/split.csv, outputs/stats.json.
`figure` writes outputs/fig_cp1_sequence.png (dark-pixel fraction vs. label).
"""
from __future__ import annotations

import argparse
import json
from pathlib import Path

import pandas as pd

from .sen12flood import build_index, official_split_leakage, split_sequences
from .stats import index_stats, pixel_stats


def cmd_ingest(a):
    out = Path(a.out)
    out.mkdir(parents=True, exist_ok=True)
    idx = build_index(a.root)
    idx.to_csv(out / "index.csv", index=False)
    split = split_sequences(idx[idx.sensor == "S1"], test_frac=a.test_frac, seed=a.seed)
    split.to_csv(out / "split.csv", index=False)
    stats = {"root": str(a.root), "index": index_stats(idx),
             "split": split.split.value_counts().to_dict(),
             "split_groups": int(split.group.nunique()),
             "official_split_leakage": official_split_leakage(idx)}
    if not a.no_pixels:
        stats["pixels_S1"] = pixel_stats(idx, n=a.n_pixels)
    (out / "stats.json").write_text(json.dumps(stats, indent=2, default=str), encoding="utf-8")
    print(json.dumps(stats, indent=2, default=str))


def cmd_figure(a):
    from .figures import cp1_figure

    out = Path(a.out)
    idx = pd.read_csv(out / "index.csv", parse_dates=["date"], dtype={"seq": str})
    path = cp1_figure(idx, out, seq_main=a.seq, thr_db=a.water_db, lang=a.lang)
    print("wrote", path)


def cmd_learn(a):
    import time

    import numpy as np

    from .features import extract_features
    from .learn import run_experiment

    out = Path(a.out)
    out.mkdir(parents=True, exist_ok=True)
    fp, hp = out / "features.csv", out / "vv_hist.npy"
    if fp.exists() and hp.exists() and not a.recompute:
        df, hists = pd.read_csv(fp, dtype={"seq": str}), np.load(hp)
    else:
        t0 = time.time()
        idx = build_index(a.root, sensors=("S1",))
        print(f"index: {int(idx.available.sum())} S1 images with files / {len(idx)} listed", flush=True)
        df, hists = extract_features(idx, n_jobs=a.jobs, verbose=5)
        df.to_csv(fp, index=False)
        np.save(hp, hists)
        print(f"features: {df.shape} in {time.time() - t0:.0f}s", flush=True)
    seeds = list(range(a.seeds))
    res = run_experiment(df, hists, seeds=seeds, test_frac=a.test_frac)
    res.round(4).to_csv(out / a.results, index=False)  # 4 zecimale: CSV-ul raportat
    s = res[res.row_type != "run"][["row_type", "method", "metric", "mean", "ci95_lo", "ci95_hi"]]
    print(s.to_string(index=False, float_format=lambda v: f"{v:.3f}"))
    off = res[(res.row_type == "run") & (res.split == "official_leaky")][["method", "bal_acc", "acc", "f1", "auc"]]
    print("\nofficial (leaky) split:\n" + off.to_string(index=False, float_format=lambda v: f"{v:.3f}"))
    print("wrote", out / a.results)


def main(argv=None):
    p = argparse.ArgumentParser(prog="h8twin")
    sub = p.add_subparsers(required=True)
    i = sub.add_parser("ingest")
    i.add_argument("--root", required=True, help="folder with S1list.json, S2list.json, 0001..0336")
    i.add_argument("--out", default="outputs")
    i.add_argument("--test-frac", type=float, default=0.2)
    i.add_argument("--seed", type=int, default=42)
    i.add_argument("--n-pixels", type=int, default=60)
    i.add_argument("--no-pixels", action="store_true")
    i.set_defaults(fn=cmd_ingest)
    f = sub.add_parser("figure")
    f.add_argument("--out", default="outputs")
    f.add_argument("--seq", default=None)
    f.add_argument("--water-db", type=float, default=-18.0)
    f.add_argument("--lang", choices=["ro", "en"], default="ro")
    f.set_defaults(fn=cmd_figure)
    l = sub.add_parser("learn")
    l.add_argument("--root", required=True)
    l.add_argument("--out", default="outputs")
    l.add_argument("--seeds", type=int, default=5)
    l.add_argument("--test-frac", type=float, default=0.2)
    l.add_argument("--jobs", type=int, default=-1)
    l.add_argument("--results", default="results_v1_H8.csv")
    l.add_argument("--recompute", action="store_true")
    l.set_defaults(fn=cmd_learn)
    a = p.parse_args(argv)
    a.fn(a)


if __name__ == "__main__":
    main()


## 1. Ingestie: statistici pe tot setul (data card)

In [ ]:
import importlib, json
import h8twin, h8twin.cli
from h8twin.cli import main
t0 = time.time()
main(['ingest', '--root', ROOT, '--out', OUT, '--n-pixels', '60'])
print(f'ingest: {time.time()-t0:.0f}s')

## 2. Învățare: trăsături, baseline-uri, metodă, 5 seed-uri

In [ ]:
t0 = time.time()
main(['learn', '--root', ROOT, '--out', OUT, '--seeds', '5', '--jobs', '-1'])
print(f'learn: {time.time()-t0:.0f}s')

## 3. Tabelul pentru slide-ul 2 și figura

In [ ]:
import pandas as pd, numpy as np
import matplotlib; import matplotlib.pyplot as plt
res = pd.read_csv(f'{OUT}/results_v1_H8.csv')
summ = res[res.row_type == 'summary']
tab = summ.pivot_table(index='method', columns='metric', values=['mean', 'ci95_lo', 'ci95_hi'], sort=False)
show = pd.DataFrame({m: [f"{tab.loc[i, ('mean', m)]:.3f} [{tab.loc[i, ('ci95_lo', m)]:.3f}, {tab.loc[i, ('ci95_hi', m)]:.3f}]" for i in tab.index]
                     for m in ['bal_acc', 'f1', 'auc']}, index=tab.index)
print(show.to_string())
print(res[res.row_type == 'paired_diff'][['method', 'mean', 'ci95_lo', 'ci95_hi']].to_string(index=False))
b = summ[summ.metric == 'bal_acc'].reset_index(drop=True)
fig, ax = plt.subplots(figsize=(8, 3.2), dpi=150)
ax.barh(b.method, b['mean'], xerr=[b['mean'] - b.ci95_lo, b.ci95_hi - b['mean']], color='#3F6EA8', capsize=3)
ax.axvline(0.5, color='grey', lw=1, ls='--'); ax.invert_yaxis(); ax.set_xlabel('balanced accuracy (IC 95 %, 5 seed-uri)')
fig.tight_layout(); fig.savefig(f'{OUT}/fig_cp3_results.png')

In [ ]:
# conținutul CSV-ului (copiat în raport) + md5, ca să se poată verifica copia din repo
import hashlib
s = open(f'{OUT}/results_v1_H8.csv').read()
print(hashlib.md5(s.encode()).hexdigest(), len(s))
print(s)